# Prepare fold BEDs

Based on [this Colab](https://colab.research.google.com/drive/1qjR2W1LpRbrb0zmGescaiUGe7Ft1k0kN).
Use **ENCFF802KJC (IMR-90)**


In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import tarfile

import pandas as pd
import pyranges as pr
from alphagenome.models.dna_client import ModelVersion, Organism
from alphagenome.data import fold_intervals

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'benchmarks/atac').is_dir())
DATA = ROOT / 'data/atac'
DATA.mkdir(parents=True, exist_ok=True)


/media/eternus1/nfs/projects/dpanchenko/BenchAG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download and read the test peaks


In [2]:
archive = DATA / 'ENCFF802KJC.tar.gz'
if not archive.exists():
    urlretrieve('https://www.encodeproject.org/files/ENCFF802KJC/@@download/', archive)

columns = ['Chromosome', 'Start', 'End', 'Name', 'Score',
           'Strand', 'Signal', 'pValue', 'qValue', 'Summit']
with tarfile.open(archive) as tar:
    with tar.extractfile('./fold_0/peaks.testset.fold_0.ENCSR200OML.bed.gz') as peaks:
        df_test = pd.read_csv(peaks, compression='gzip', sep='\t', header=None, names=columns)
print(f'Original test peaks: {len(df_test)}')


Original test peaks: 59007


## 2. Remove training overlaps and save both BEDs

For each AG model, expand training intervals to 1 Mb and remove overlapping peaks.


In [3]:
for version, filename in [(ModelVersion.FOLD_0, 'fold0_test_regions.bed'),
                          (ModelVersion.FOLD_1, 'fold1_test_regions.bed')]:
    df_train = fold_intervals.get_fold_intervals(
        model_version=version,
        organism=Organism.HOMO_SAPIENS,
        subset=fold_intervals.Subset.TRAIN,
        example_regions_path=str(ROOT / 'benchmarks/atac/data/sequences_human.bed.gz'),
    ).rename(columns={'chromosome': 'Chromosome', 'start': 'Start', 'end': 'End'}).copy()

    center = (df_train.Start + df_train.End) // 2
    df_train['Start'] = center - 2**19
    df_train['End'] = df_train['Start'] + 2**20

    filtered = pr.PyRanges(df_test).overlap(pr.PyRanges(df_train), invert=True).df[columns]
    # Match the colleague's text sorting of BED lines.
    lines = filtered.to_csv(sep='\t', header=False, index=False).splitlines(keepends=True)
    (DATA / filename).write_text(''.join(sorted(lines)))
    print(f'{filename}: {len(filtered)} peaks')


fold0_test_regions.bed: 11860 peaks


fold1_test_regions.bed: 13812 peaks
